In [ ]:
from vpython import canvas, graph, gcurve, wtext, button, slider, sphere, vector, color, rate
import random

# 1. GLOBAL SYSTEM VARIABLES
is_running = True
grid_width = 9   
grid_height = 9  
spacing = 1.3 
temperature = 300  

# Filter Structure Matrix
grid_matrix = {}
filter_atoms = []

# Gas Lists and Telemetry Initialization
all_gas_molecules = []
blue_total = blue_passed = red_total = red_blocked = 0
dt = 0.02
frame_counter = 0
time_elapsed = 0  

# Chemical Gas Configurations
gas_A = {"radius": 0.2, "speed": 3.0, "color": color.cyan, "name": "Nitrogen (N2)", "mass": 28}
gas_B = {"radius": 0.6, "speed": 1.5, "color": color.red,  "name": "Soot / Smoke", "mass": 200}
current_mode_name = "Air Purification (Smoke vs N2)"

# Action History Stacks (Undo / Redo)
undo_stack = []  
redo_stack = []  

# --- UI INTERACTIVE INTERACTION FUNCTIONS ---
def set_mode_air(b):
    global gas_A, gas_B, current_mode_name
    gas_A = {"radius": 0.2, "speed": 3.0, "color": color.cyan, "name": "Nitrogen (N2)", "mass": 28}
    gas_B = {"radius": 0.6, "speed": 1.5, "color": color.red,  "name": "Soot / Smoke", "mass": 200}
    current_mode_name = "Air Purification (Smoke vs N2)"; reset_statistics()

def set_mode_hydrogen(b):
    global gas_A, gas_B, current_mode_name
    gas_A = {"radius": 0.15, "speed": 4.0, "color": color.yellow, "name": "Hydrogen (H2)", "mass": 2}
    gas_B = {"radius": 0.40, "speed": 2.2, "color": color.orange, "name": "Methane (CH4)", "mass": 16}
    current_mode_name = "Hydrogen Extraction (H2 from CH4)"; reset_statistics()

def set_mode_isotopes(b):
    global gas_A, gas_B, current_mode_name
    gas_A = {"radius": 0.30, "speed": 2.5, "color": color.green, "name": "Light Isotope", "mass": 349}
    gas_B = {"radius": 0.36, "speed": 2.4, "color": color.magenta, "name": "Heavy Isotope", "mass": 352}
    current_mode_name = "Isotope Separation (High Precision)"; reset_statistics()

def reset_statistics():
    global blue_total, blue_passed, red_total, red_blocked, all_gas_molecules, time_elapsed
    for gas in all_gas_molecules: gas.visible = False
    all_gas_molecules.clear()
    blue_total = blue_passed = red_total = red_blocked = 0
    time_elapsed = 0
    efficiency_curve.delete()
    throughput_curve.delete()

def toggle_simulation(b):
    global is_running; is_running = not is_running
    if is_running: b.text = "⏸️ PAUSE & EDIT STRUCTURE"
    else: b.text = "▶️ RUN MOLECULAR FLOW"

def change_width(s): 
    global grid_width; grid_width = int(s.value)
    width_text.text = f" <b>{grid_width}</b> atoms"
    apply_custom_size()

def change_height(s): 
    global grid_height; grid_height = int(s.value)
    height_text.text = f" <b>{grid_height}</b> atoms"
    apply_custom_size()

def change_temperature(s):
    global temperature; temperature = int(s.value)
    temp_text.text = f" <b>{temperature} K</b>"

def click_action(evt):
    if not is_running:
        clicked_object = sim_canvas.mouse.pick
        if clicked_object and clicked_object in filter_atoms:
            coord = clicked_object.grid_coord
            undo_stack.append(coord)
            redo_stack.clear()
            clicked_object.visible = False
            filter_atoms.remove(clicked_object)
            if coord in grid_matrix: del grid_matrix[coord]

def undo_action(b):
    if not is_running and len(undo_stack) > 0:
        coord = undo_stack.pop(); redo_stack.append(coord); y, z = coord
        half_w = grid_width / 2; half_h = grid_height / 2
        pos_vector = vector(0, (y - half_h + 0.5) * spacing, (z - half_w + 0.5) * spacing)
        restored_atom = sphere(canvas=sim_canvas, pos=pos_vector, radius=0.5, color=color.gray(0.6))
        restored_atom.grid_coord = coord
        filter_atoms.append(restored_atom); grid_matrix[coord] = restored_atom

def redo_action(b):
    if not is_running and len(redo_stack) > 0:
        coord = redo_stack.pop(); undo_stack.append(coord)
        if coord in grid_matrix:
            atom_to_remove = grid_matrix[coord]; atom_to_remove.visible = False
            filter_atoms.remove(atom_to_remove); del grid_matrix[coord]

def apply_custom_size():
    global blue_total, blue_passed, red_total, red_blocked, all_gas_molecules, half_w, half_h, time_elapsed
    for atom in filter_atoms: atom.visible = False
    filter_atoms.clear()
    for gas in all_gas_molecules: gas.visible = False
    all_gas_molecules.clear()
    undo_stack.clear(); redo_stack.clear(); grid_matrix.clear()
    blue_total = blue_passed = red_total = red_blocked = 0; time_elapsed = 0
    efficiency_curve.delete(); throughput_curve.delete()
    half_w = grid_width / 2; half_h = grid_height / 2
    
    for y in range(0, grid_height):
        for z in range(0, grid_width):
            pos_vector = vector(0, (y - half_h + 0.5) * spacing, (z - half_w + 0.5) * spacing)
            atom = sphere(canvas=sim_canvas, pos=pos_vector, radius=0.5, color=color.gray(0.6))
            atom.grid_coord = (y, z) 
            filter_atoms.append(atom); grid_matrix[(y, z)] = atom

# 2. 3D CANVAS INITIALIZATION
sim_canvas = canvas(title="<h3>MolFlow v2.2 — Segmented UI Column Layout</h3>", 
                    width=700, height=380, center=vector(0, 0, 0), background=color.gray(0.15))
sim_canvas.bind('mousedown', click_action)


# --- СЕГМЕНТИРОВАННЫЙ ИНТЕРФЕЙС В СТОЛБИК ---
# БЛОК 1: Основные кнопки управления симуляцией
wtext(text="<div style='font-family: \"Segoe UI\", sans-serif; width: 680px; margin-top: 15px;'>")
wtext(text="<fieldset style='border: 1px solid #34495e; padding: 12px; border-radius: 6px; margin-bottom: 10px;'>")
wtext(text="<legend style='font-weight: bold; color: #2c3e50;'>🎮 Core Engine Controls</legend>")
control_button = button(bind=toggle_simulation, text="⏸️ PAUSE & EDIT STRUCTURE")
wtext(text=" &nbsp; ")
undo_button = button(bind=undo_action, text="↩️ Undo")
wtext(text=" ")
redo_button = button(bind=redo_action, text="↪️ Redo")
wtext(text="</fieldset>")

# БЛОК 2: Сегмент геометрических настроек мембраны
wtext(text="<fieldset style='border: 1px solid #34495e; padding: 12px; border-radius: 6px; margin-bottom: 10px;'>")
wtext(text="<legend style='font-weight: bold; color: #2c3e50;'>📐 Membrane Geometry</legend>")
wtext(text="Width Scale: <br>")
width_slider = slider(min=1, max=20, value=9, step=1, bind=change_width)
width_text = wtext(text=f" <b>{grid_width}</b> atoms")
wtext(text="<br><br>Height Scale: <br>")
height_slider = slider(min=1, max=20, value=9, step=1, bind=change_height)
height_text = wtext(text=f" <b>{grid_height}</b> atoms")
wtext(text="</fieldset>")

# БЛОК 3: Сегмент термодинамики (температура)
wtext(text="<fieldset style='border: 1px solid #34495e; padding: 12px; border-radius: 6px; margin-bottom: 10px;'>")
wtext(text="<legend style='font-weight: bold; color: #2c3e50;'>Thermodynamics Parameter</legend>")
wtext(text="System Temperature: <br>")
temp_slider = slider(min=0, max=600, value=300, step=10, bind=change_temperature)
temp_text = wtext(text=f" <b>{temperature} K</b>")
wtext(text="</fieldset>")

# БЛОК 4: Сегмент выбора химических веществ
wtext(text="<fieldset style='border: 1px solid #34495e; padding: 12px; border-radius: 6px; margin-bottom: 10px;'>")
wtext(text="<legend style='font-weight: bold; color: #2c3e50;'>🧪 Chemical Gas Injector</legend>")
btn_mode1 = button(bind=set_mode_air, text="💨 Air Purification (Smoke)")
wtext(text="<br><br>")
btn_mode2 = button(bind=set_mode_hydrogen, text="🔋 Hydrogen Extraction (H2)")
wtext(text="<br><br>")
btn_mode3 = button(bind=set_mode_isotopes, text="⚛️ Isotope Separation")
wtext(text="</fieldset>")
wtext(text="</div>")


# 3. LIVE KINETICS GRAPH INITIALIZATION
kinetics_graph = graph(title="<b>Filtration Kinetics vs Time</b>", 
                       xtitle="Time (Seconds)", ytitle="Performance (%)", 
                       width=700, height=220, min=0, max=100, background=color.gray(0.95))

efficiency_curve = gcurve(graph=kinetics_graph, color=color.red, label="Efficiency (Blocked Waste)")
throughput_curve = gcurve(graph=kinetics_graph, color=color.cyan, label="Throughput (Passed Gas)")

print("\n\n")
stats_panel = wtext(text="Initializing Interface...")

apply_custom_size()

# 4. MAIN SIMULATION LOOP
while True:
    rate(60)
    if is_running:
        frame_counter += 1
        time_elapsed += dt
        
        # Generator
        if frame_counter % 15 == 0:
            spawn_limit_y = half_h * spacing
            spawn_limit_z = half_w * spacing
            start_pos = vector(-8, random.uniform(-spawn_limit_y, spawn_limit_y), random.uniform(-spawn_limit_z, spawn_limit_z))
            
            if random.random() > 0.5:
                new_gas = sphere(canvas=sim_canvas, pos=start_pos, radius=gas_A["radius"], color=gas_A["color"])
                new_gas.velocity = vector(random.uniform(gas_A["speed"]*0.8, gas_A["speed"]*1.2), random.uniform(-0.2, 0.2), random.uniform(-0.2, 0.2))
                new_gas.type = "blue"; new_gas.mass = gas_A["mass"]
                blue_total += 1
            else:
                new_gas = sphere(canvas=sim_canvas, pos=start_pos, radius=gas_B["radius"], color=gas_B["color"])
                new_gas.velocity = vector(random.uniform(gas_B["speed"]*0.8, gas_B["speed"]*1.2), random.uniform(-0.1, 0.1), random.uniform(-0.1, 0.1))
                new_gas.type = "red"; new_gas.mass = gas_B["mass"]
                red_total += 1
            all_gas_molecules.append(new_gas)
            
        for gas in all_gas_molecules[:]:
            # Thermal Chaos
            if temperature > 0:
                chaos_factor = (temperature / 100) / (gas.mass ** 0.5)
                thermal_noise = vector(0, 
                                       random.uniform(-chaos_factor, chaos_factor), 
                                       random.uniform(-chaos_factor, chaos_factor))
                gas.velocity = gas.velocity + thermal_noise
            
            gas.pos = gas.pos + gas.velocity * dt
            
            # Collisions with filter atoms
            for atom in filter_atoms:
                distance = (gas.pos - atom.pos).mag
                min_dist = gas.radius + atom.radius
                if distance < min_dist:
                    normal = (gas.pos - atom.pos).norm()
                    gas.velocity = gas.velocity - 2 * gas.velocity.dot(normal) * normal
                    gas.pos = atom.pos + normal * min_dist
                    
            # Out of bounds and scoring logic
            if gas.type == "blue" and gas.pos.x > 6:
                blue_passed += 1; gas.visible = False; all_gas_molecules.remove(gas)
            elif gas.type == "red" and gas.pos.x < -9:
                red_blocked += 1; gas.visible = False; all_gas_molecules.remove(gas)
            elif gas.pos.x > 8 or gas.pos.x < -10:
                gas.visible = False; all_gas_molecules.remove(gas)

        # Telemetry processing
        efficiency = (red_blocked / red_total * 100) if red_total > 0 else 0
        flow_rate = (blue_passed / blue_total * 100) if blue_total > 0 else 0

        # Graph plot update
        if frame_counter % 5 == 0:
            efficiency_curve.plot(time_elapsed, efficiency)
            throughput_curve.plot(time_elapsed, flow_rate)

    # 5. LUXURY DARK-THEME DASHBOARD (STYLING INTTACT)
    status_text = "<span style='color: #2ecc71; font-weight: bold;'>FLOW ACTIVE</span>" if is_running else "<span style='color: #e67e22; font-weight: bold;'>STRUCTURE EDIT MODE</span>"
    
    stats_panel.text = (
        f"<div style='border: none; padding: 20px; background-color: #2c3e50; color: #ecf0f1; "
        f"font-family: \"Segoe UI\", sans-serif; width: 660px; border-radius: 10px; "
        f"box-shadow: 0 4px 15px rgba(0,0,0,0.3); margin-top: 15px;'>"
        f"<h3 style='margin-top: 0; color: #3498db; border-bottom: 2px solid #34495e; padding-bottom: 10px;'>📊 MolFlow Telemetry Core v2.2</h3>"
        f"<b>Current Chemical Mode:</b> <span style='color: #f1c40f; font-weight: bold;'>{current_mode_name}</span><br>"
        f"<b>System Status:</b> {status_text}<br>"
        f"<b>System Temperature:</b> <span style='color: #e67e22; font-weight: bold;'>{temperature} K</span><br>"
        f"<b>Active Membrane Geometry:</b> {grid_width}w x {grid_height}h atoms<br>"
        f"<div style='margin: 15px 0; border-top: 1px solid #34495e;'></div>"
        f"<b>BLOCKED {gas_B['name'].upper()}:</b> {red_blocked} / {red_total}<br>"
        f"🎯 <span style='color: #e74c3c; font-weight: bold; font-size: 16px;'>FILTRATION EFFICIENCY: {efficiency:.1f}%</span><br>"
        f"<div style='margin: 10px 0;'></div>"
        f"<b>PASSED {gas_A['name'].upper()}:</b> {blue_passed} / {blue_total}<br>"
        f"⚡ <span style='color: #1abc9c; font-weight: bold; font-size: 16px;'>THROUGHPUT CAPACITY: {flow_rate:.1f}%</span>"
        f"</div>"
    )
